# 3. Interpretabilidad con LIME

Los modelos de árboles dan importancias **globales** (qué features usa el modelo en general), pero no explican una predicción **individual**: ¿por qué este cliente concreto recibió 0.87 de probabilidad de churn?

LIME (Local Interpretable Model-agnostic Explanations) responde esa pregunta entrenando un modelo lineal simple alrededor de la observación de interés, perturbando ligeramente sus features y viendo cómo cambia la predicción. El resultado es una lista de features con su contribución (positiva o negativa) a esa predicción específica.

**Aplicaremos LIME a 3 casos representativos**:
1. Un cliente con alta probabilidad de churn (verdadero positivo).
2. Un cliente fiel que el modelo identifica correctamente como retenido.
3. Un caso en la frontera (probabilidad cercana a 0.5).

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from pathlib import Path

import sys
sys.path.append("..")
from app.preprocessing import (
    ALL_FEATURES, NUMERIC_FEATURES, CATEGORICAL_FEATURES,
    TARGET_COLUMN, build_feature_frame, clean_total_charges,
)

import lime
import lime.lime_tabular

RANDOM_STATE = 42

## 3.1 Recuperar el modelo y los datos

In [ ]:
artifact = joblib.load("../app/model.joblib")
pipeline = artifact["model"]
print(f"Modelo cargado: {artifact['name']}")
print(f"Métricas guardadas: {artifact.get('metrics')}")

In [ ]:
df = pd.read_csv("../data/telco_churn.csv").drop(columns=["customerID"])
df = clean_total_charges(df)
y = (df[TARGET_COLUMN] == "Yes").astype(int)
X = build_feature_frame(df.drop(columns=[TARGET_COLUMN]))

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE,
)
print(f"Test set: {X_test.shape}")

## 3.2 Configurar LimeTabularExplainer

LIME espera arrays numéricos. Como nuestro pipeline ya hace one-hot internamente, tenemos dos opciones:
1. Aplicar LIME **sobre los datos crudos** (DataFrame con strings) y pasar una `predict_fn` que envuelva el pipeline. Eso nos obliga a indicar a LIME qué columnas son categóricas.
2. Aplicar LIME **sobre los datos ya transformados**, con el preprocessor extraído del pipeline.

Vamos con la opción **(2)** porque LIME explica mejor cuando trabaja con la representación numérica final que ve el modelo.

In [ ]:
# Extraer el preprocesador y transformar
preprocessor = pipeline.named_steps["preprocessor"]
model_only   = pipeline.named_steps["model"]

X_train_arr = preprocessor.transform(X_train)
X_test_arr  = preprocessor.transform(X_test)
feature_names = list(preprocessor.get_feature_names_out())
print(f"Features tras preprocesar: {len(feature_names)}")

In [ ]:
explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train_arr,
    feature_names=feature_names,
    class_names=["No Churn", "Churn"],
    mode="classification",
    discretize_continuous=True,
    random_state=RANDOM_STATE,
)

## 3.3 Función helper para mostrar explicaciones

In [ ]:
def explain_case(idx, X_arr, X_raw, y_true, num_features=10):
    """Genera y muestra la explicación LIME para una fila del test."""
    proba = model_only.predict_proba(X_arr[idx:idx+1])[0]
    real_label = "Churn" if y_true.iloc[idx] == 1 else "No Churn"

    print(f"\nCaso #{idx}")
    print(f"  Real:        {real_label}")
    print(f"  P(No Churn): {proba[0]:.3f}")
    print(f"  P(Churn):    {proba[1]:.3f}")
    print("\nDatos originales del cliente:")
    print(X_raw.iloc[idx].to_dict())

    exp = explainer.explain_instance(
        data_row=X_arr[idx],
        predict_fn=model_only.predict_proba,
        num_features=num_features,
    )
    fig = exp.as_pyplot_figure()
    fig.tight_layout()
    plt.show()
    return exp

## 3.4 Caso 1 — Cliente con alta probabilidad de churn

In [ ]:
# Encontramos el cliente con mayor probabilidad de churn entre los reales positivos
probas = model_only.predict_proba(X_test_arr)[:, 1]
mask_pos = (y_test.values == 1)
top_idx = np.where(mask_pos)[0][np.argmax(probas[mask_pos])]
exp1 = explain_case(top_idx, X_test_arr, X_test, y_test)

**Análisis Caso 1**:
- Las **barras naranjas/rojas** del lado derecho empujan hacia la clase **Churn**: típicamente verás `Contract_Month-to-month=1`, `tenure ≤ X` (con X bajo, ej. ≤ 9), `InternetService_Fiber optic=1`, `OnlineSecurity_No=1`.
- Las **barras del lado izquierdo (verde/azul)**, si las hay, empujan hacia "No Churn" — pero en un caso de alta probabilidad de churn habrá pocas o ninguna.
- Esta lectura es local: para *otro* cliente con churn alto, el conjunto de razones puede variar (ej. uno por contrato corto, otro por cargos altos).

## 3.5 Caso 2 — Cliente fiel correctamente clasificado

In [ ]:
mask_neg = (y_test.values == 0)
low_idx = np.where(mask_neg)[0][np.argmin(probas[mask_neg])]
exp2 = explain_case(low_idx, X_test_arr, X_test, y_test)

**Análisis Caso 2**:
- Aquí esperamos ver lo opuesto: features que **empujan a No Churn** dominan. Casi siempre `Contract_Two year=1`, `tenure` alto (> 50 meses), `OnlineSecurity_Yes=1`, `TechSupport_Yes=1`.
- Estos clientes son "anclas" — el modelo está muy seguro de que se quedarán. Para una estrategia de retención, no son prioridad.

## 3.6 Caso 3 — Cliente en la frontera (decisión incierta)

In [ ]:
# Cliente más cercano a la probabilidad 0.5
borderline_idx = int(np.argmin(np.abs(probas - 0.5)))
exp3 = explain_case(borderline_idx, X_test_arr, X_test, y_test)

**Análisis Caso 3**:
- En un cliente borderline veremos una mezcla de barras hacia ambos lados. La predicción final es esencialmente una pulseada entre señales contradictorias.
- **Estos son los casos más valiosos para el negocio**: probabilidades cerca de 0.5 son los clientes en los que una intervención de retención (descuento, llamada, mejora de servicio) puede tener el mayor ROI, porque pequeños cambios en las features los pueden mover hacia "No Churn".
- También son los casos donde el modelo está menos seguro y donde *vale la pena revisar manualmente* antes de tomar acción comercial.

## 3.7 Lecturas y limitaciones de LIME

LIME es útil pero tiene puntos a tener presentes:

1. **Es local**, no global. Una explicación válida para un cliente no se generaliza a la población.
2. **Es estocástico**: LIME perturba aleatoriamente alrededor del punto. Distintas corridas pueden dar pesos ligeramente distintos. Por eso fijamos `random_state`.
3. **El modelo lineal sustituto** que LIME ajusta puede ser mala aproximación si la frontera de decisión local es muy curva.
4. **Discretización de continuas** (`discretize_continuous=True`) ayuda a leer los resultados pero pierde resolución.
5. **No es causal**: LIME explica qué features mueven la predicción, no qué causa el churn en la realidad.

**Alternativas y complementos** que podrías considerar después:
- **SHAP**: explicaciones basadas en valores de Shapley con mejor base teórica y consistencia.
- **Partial Dependence Plots / ICE**: efecto marginal de una feature sobre la predicción.
- **Counterfactuals** (ej. DiCE): "qué tendría que cambiar este cliente para que la predicción se invierta".

## 3.8 Resumen del flujo del proyecto

✅ **Notebook 1**: EDA y decisiones de preprocesamiento.
✅ **Notebook 2**: Pipeline con 4 modelos, GridSearchCV con CV estratificada, evaluación, selección y persistencia.
✅ **Notebook 3**: Interpretabilidad con LIME para 3 casos representativos.

**Siguiente paso fuera del notebook**: la API en `app/api.py` levanta el modelo y lo sirve por HTTP, todo dentro de un contenedor Docker. Ver `README.md`.